# Model Checker

## Load Data and Base Models

In [1]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/menura-gradient_boosting_classifier/preprocessed_student_depression(GBC-model).csv")

In [3]:
X = df.drop(columns=["Depression"])
y = df["Depression"]

print("X shape:", X.shape, " , y shape:", y.shape)

X shape: (27867, 12)  , y shape: (27867,)


In [6]:
dt_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/dunith_decision_tree/decision_tree_model.joblib")
rf_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211009_Themiya_random_forrest/random_forest_student_depression.joblib")
svm_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211011_kaveesha_svm_model/best_svm_model.joblib")
gb_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/menura-gradient_boosting_classifier/gbc_model.joblib")
lr_model = joblib.load("/workspaces/CS_Group07_StudentDepressionDetection/notebooks/Rushani_Logistic_Regression/final_logistic_model.joblib")

models = {
    'DecisionTree': dt_model,
    'RandomForest': rf_model,
    'SVM': svm_model,
    'GradientBoosting': gb_model,
    'LogisticRegression': lr_model
}

## Check predict() Availability

In [7]:
for name, m in models.items():
    try:
        preds = m.predict(X)
        print(f"{name:4s} -> predict() OK")
    except Exception as e:
        print(f"{name:4s} -> predict() FAILED: {e}")

DecisionTree -> predict() OK
RandomForest -> predict() OK
SVM  -> predict() OK
GradientBoosting -> predict() OK
LogisticRegression -> predict() OK


## Check predict_proba() Availability

In [8]:
for name, m in models.items():
    try:
        _ = m.predict_proba(X)
        print(f"{name:4s} -> predict_proba() OK")
    except Exception as e:
        print(f"{name:4s} -> NO predict_proba(): {e}")

DecisionTree -> predict_proba() OK
RandomForest -> predict_proba() OK
SVM  -> predict_proba() OK
GradientBoosting -> predict_proba() OK
LogisticRegression -> predict_proba() OK


## Check Predictions Shapes

In [9]:
for name, m in models.items():
    try:
        preds = m.predict(X)
        print(f"{name:4s} -> {preds.shape}")
    except:
        print(f"{name:4s} -> shape check FAILED")


DecisionTree -> (27867,)
RandomForest -> (27867,)
SVM  -> (27867,)
GradientBoosting -> (27867,)
LogisticRegression -> (27867,)


## Correlation Between Predictions

In [10]:
pred_dict = {}

for name, m in models.items():
    try:
        pred_dict[name] = m.predict(X)
    except:
        print(f"{name} FAILED prediction; skipping.")

pred_df = pd.DataFrame(pred_dict)
print(pred_df.corr())

                    DecisionTree  RandomForest       SVM  GradientBoosting  \
DecisionTree            1.000000      0.822803  0.832694          0.844596   
RandomForest            0.822803      1.000000  0.820317          0.831529   
SVM                     0.832694      0.820317  1.000000          0.918227   
GradientBoosting        0.844596      0.831529  0.918227          1.000000   
LogisticRegression      0.825882      0.814252  0.978168          0.910448   

                    LogisticRegression  
DecisionTree                  0.825882  
RandomForest                  0.814252  
SVM                           0.978168  
GradientBoosting              0.910448  
LogisticRegression            1.000000  


## Model Disagreement (Diversity)

In [11]:
names = list(pred_df.columns)
pred_list = [pred_df[n].values for n in names]

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        disagree = np.mean(pred_list[i] != pred_list[j])
        print(f"{names[i]} vs {names[j]} -> disagreement: {disagree:.3f}")

DecisionTree vs RandomForest -> disagreement: 0.085
DecisionTree vs SVM -> disagreement: 0.082
DecisionTree vs GradientBoosting -> disagreement: 0.074
DecisionTree vs LogisticRegression -> disagreement: 0.086
RandomForest vs SVM -> disagreement: 0.088
RandomForest vs GradientBoosting -> disagreement: 0.081
RandomForest vs LogisticRegression -> disagreement: 0.091
SVM vs GradientBoosting -> disagreement: 0.041
SVM vs LogisticRegression -> disagreement: 0.011
GradientBoosting vs LogisticRegression -> disagreement: 0.045


## Individual Model Accuracy

In [12]:
for name, preds in pred_dict.items():
    acc = accuracy_score(y, preds)
    print(f"{name:4s} -> accuracy: {acc:.4f}")

DecisionTree -> accuracy: 0.8550
RandomForest -> accuracy: 0.9240
SVM  -> accuracy: 0.8466
GradientBoosting -> accuracy: 0.8543
LogisticRegression -> accuracy: 0.8442


## Joblib Model Inspecter

### Helper

In [16]:
model_paths = {
    "Decision Tree": "/workspaces/CS_Group07_StudentDepressionDetection/notebooks/dunith_decision_tree/decision_tree_model.joblib",
    "Random Forest": "/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211009_Themiya_random_forrest/random_forest_student_depression.joblib",
    "SVM": "/workspaces/CS_Group07_StudentDepressionDetection/notebooks/fc211011_kaveesha_svm_model/best_svm_model.joblib",
    "Gradient Boosting": "/workspaces/CS_Group07_StudentDepressionDetection/notebooks/menura-gradient_boosting_classifier/gbc_model.joblib",
    "Logistic Regression": "/workspaces/CS_Group07_StudentDepressionDetection/notebooks/Rushani_Logistic_Regression/final_logistic_model.joblib",
}

In [17]:
def inspect_model(model, name):
    print("=" * 70)
    print(f"🔍 Inspecting {name}")
    print("=" * 70)
    print("Type:", type(model))
    print()

    # --- General parameters ---
    if hasattr(model, "get_params"):
        print("⚙️ Parameters:")
        for k, v in model.get_params().items():
            print(f"  {k}: {v}")
        print()

    # --- Coefficients (for linear models) ---
    if hasattr(model, "coef_"):
        print("📊 Coefficients shape:", model.coef_.shape)
        print("📊 Coefficients (first 10):", model.coef_.ravel()[:10])
        print()

    # --- Intercept ---
    if hasattr(model, "intercept_"):
        print("📈 Intercept:", model.intercept_)
        print()

    # --- Feature importances (for trees/ensembles) ---
    if hasattr(model, "feature_importances_"):
        print("🌲 Feature Importances (Top 10):")
        importances = model.feature_importances_
        top_features = importances.argsort()[-10:][::-1]
        print(importances[top_features])
        print()

    # --- Support vectors (for SVMs) ---
    if hasattr(model, "support_vectors_"):
        print("📉 Support Vectors shape:", model.support_vectors_.shape)
        print()

    # --- Estimators (for ensemble models) ---
    if hasattr(model, "estimators_"):
        print("🧠 Number of Base Estimators:", len(model.estimators_))
        print()

    # --- If it's a pipeline ---
    from sklearn.pipeline import Pipeline
    if isinstance(model, Pipeline):
        print("🔗 This is a Pipeline. Steps included:")
        for step_name, step_obj in model.named_steps.items():
            print(f"  - {step_name}: {type(step_obj)}")
        print()

    print("✅ Inspection complete for", name, "\n")

### Inspect Models

In [18]:
for name, path in model_paths.items():
    try:
        model = joblib.load(path)
        inspect_model(model, name)
    except Exception as e:
        print(f"❌ Error loading {name}: {e}\n")

🔍 Inspecting Decision Tree
Type: <class 'sklearn.pipeline.Pipeline'>

⚙️ Parameters:
  memory: None
  steps: [('preprocess', ColumnTransformer(transformers=[('num', 'passthrough',
                                 ['Age', 'Academic_Pressure', 'CGPA',
                                  'Study_Satisfaction', 'Study_Hours',
                                  'Financial_Stress']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Gender', 'Sleep_Duration', 'Dietary_Habits',
                                  'Degree', 'Suicidal_Thoughts',
                                  'Mental_Illness_History'])])), ('model', DecisionTreeClassifier(max_depth=9, min_samples_leaf=3, random_state=42))]
  transform_input: None
  verbose: False
  preprocess: ColumnTransformer(transformers=[('num', 'passthrough',
                                 ['Age', 'Academic_Pressure', 'CGPA',
                                  'Study_Satisfaction', 'Study_Hours